In [ ]:
import pandas as pd
from pathlib import Path
from pathlib import Path
from gbmcollaboration import CHUVDataCenter, UKERDataCenter
from gbmcollaboration.utils.infercnv_utils_fixed import InferCNVAnalyzer
import gc
import scanpy as sc
import numpy as np

def load_visium_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    list_sample_id=None,
    normalization="log_norm_count",
    radius=1,
    deconvolution_slot_name="deconv_cohort_level_4_2steps",
    ):
    """Load entire Visium cohort as anndata."""

    # Define data centers and common loading parameters
    data_centers = [CHUVDataCenter, UKERDataCenter]

    # Load all data
    loaded_objects = []
    for data_center in data_centers:
        print('Loading',data_center)

        loaded_objects.append(data_center.load_spatial(
            exclude_low_quality_visium=exclude_low_quality_visium,
            exclude_idh1_mut=exclude_idh1_mut,
            list_sample_id=list_sample_id,
            normalization=normalization,
            radius=radius,
            deconvolution_slot_name=deconvolution_slot_name,
        ))

    # Concatenate the dataframes
    counts_df = pd.concat([obj.genecounts_df for obj in loaded_objects])
    obs_df = pd.concat([obj.celltype_df for obj in loaded_objects])

    del loaded_objects
    gc.collect()

    # Align obs to counts and create the 'sample' column
    obs_df = obs_df.loc[counts_df.index]
    obs_df['sample'] = obs_df.index.get_level_values(0)

    # Flatten the MultiIndex for AnnData compatibility
    flat_index = counts_df.index.map(lambda idx: f"{idx[0]}_{idx[1]}")
    obs_df.index = flat_index
    counts_df.index = flat_index

    counts_df.sum(axis=0).sort_values()

    # Create the final AnnData object
    adata = sc.AnnData(
        obs=obs_df,
        var=pd.DataFrame(index=counts_df.columns),
        layers={normalization:counts_df.values}
    )

    del counts_df, obs_df
    gc.collect()

    # --- Output ---
    print(" Final adata created:")
    print(" • shape:", adata.shape)
    print(" • layers:", list(adata.layers.keys()))
    print(" • obs columns:", adata.obs.columns.tolist())
    print(" • samples:", adata.obs['sample'].unique().tolist())

    return adata


In [ ]:
list(RESULTS_DIR_COHORT.iterdir())

In [ ]:
# %% Params
RESULTS_DIR_COHORT = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/visium/infercnv")
RESULTS_DIR_PER_SAMPLE = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/visium/infercnv_per_sample")
METACONTROL_PATH = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/meta_control_v2.h5ad")
CELL_TYPE_KEY = "deconv_cohort_level_4"
BATCH_KEY = 'sample'
REFERENCE_COLUMN = "malignant_status"

# Load data
adata = load_visium_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    list_sample_id=None,
    normalization="raw_count",
    radius=1,
    deconvolution_slot_name=CELL_TYPE_KEY,
)
adata.obsm['spatial']=adata.obs[['imagerow', 'imagecol']].values

# Load meta-control spots info
adata_ctrl = sc.read_h5ad(METACONTROL_PATH)
healthy_spots = adata_ctrl.obs_names

adata.obs[REFERENCE_COLUMN] = 'malignant'
adata.obs.loc[healthy_spots,REFERENCE_COLUMN] = 'healthy'
print(f"Using {len(healthy_spots)} healthy spots")


# load infercnv results across the cohort
df_infercnv_cohort = pd.read_csv(RESULTS_DIR_COHORT / 'tumor_clone_annotation.csv',index_col=0)
adata.obs[['cnv_leiden_cohort','cnv_score_cohort']] = df_infercnv_cohort
adata.obs['cnv_leiden_cohort']=adata.obs['cnv_leiden_cohort'].astype(str)

# load infercnv results per sample
adata.obs['cnv_leiden_per_sample'] = ''
adata.obs['cnv_score_per_sample'] = np.nan
for sample_dir in RESULTS_DIR_PER_SAMPLE.iterdir():
    sample_name = sample_dir.stem
    df_sample = pd.read_csv(sample_dir / 'tumor_clone_annotation.csv',index_col=0)
    adata.obs.loc[adata.obs[BATCH_KEY]==sample_name,'cnv_leiden_per_sample'] = df_sample['cnv_leiden']
    adata.obs.loc[adata.obs[BATCH_KEY]==sample_name,'cnv_score_per_sample'] = df_sample['cnv_score']

adata.obs['cnv_leiden_per_sample']=adata.obs['cnv_leiden_per_sample'].astype(int).astype(str)


# plot
sample_name = 'CH_G_166a_vis'
adata_sample = adata[adata.obs[BATCH_KEY]==sample_name]
sc.pl.spatial(adata_sample,color=['cnv_leiden_cohort','cnv_leiden_per_sample'],spot_size=10)

In [ ]:
adata.obs[['cnv_leiden','cnv_score']]